In [1]:
# download data
import xml.etree.ElementTree as ET
import requests
import os
import concurrent.futures

def fetch_xml(xml_url):
    response = requests.get(xml_url)
    if response.status_code != 200:
        print(f"Failed to fetch XML: {response.status_code}")
        return None
    return response.content

def parse_xml(xml_content):
    root = ET.fromstring(xml_content)
    namespace = {'s3': 'http://doc.s3.amazonaws.com/2006-03-01'}
    base_url = "https://storage.googleapis.com/quickdraw_dataset/"

    file_urls = []
    for content in root.findall(".//s3:Contents", namespace):
        key = content.find("s3:Key", namespace).text
        if key.startswith("sketchrnn/") and key.endswith(".npz") and not key.endswith(".full.npz"):
            file_urls.append(base_url + key)
    return file_urls

def download_file(file_url, download_folder):
    file_path = os.path.join(download_folder, os.path.basename(file_url))
    if os.path.exists(file_path):
        print(f"Already exists: {file_path}")
        return
    print(f" Downloading: {file_url}")
    response = requests.get(file_url)
    if response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(response.content)
    else:
        print(f"Failed to download: {file_url}")

def download_npy_files(xml_url, download_folder):
    if not os.path.exists(download_folder):
        os.makedirs(download_folder)

    xml_content = fetch_xml(xml_url)
    if xml_content is None:
        return

    file_urls = parse_xml(xml_content)

    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        executor.map(lambda url: download_file(url, download_folder), file_urls)

xml_url = "https://storage.googleapis.com/quickdraw_dataset?prefix=sketchrnn/"
xml_content = fetch_xml(xml_url)
file_urls = parse_xml(xml_content)

In [3]:
import pandas as pd
import numpy as np
files_to_get=pd.Series([
    "baseball", "basketball", "soccer ball", "wheel", "cookie", "donut", "moon", "clock", "pizza",
    "cup", "mug", "coffee cup", "wine glass", "vase", "wine bottle", "teapot",
    "pencil", "marker", "toothbrush", "screwdriver", "fork", "knife",
    "car", "truck", "pickup truck", "bus", "school bus", "van",
    "sailboat", "speedboat", "canoe", "cruise ship",
    "microwave", "oven", "toaster", "dishwasher", "washing machine",
    "chair", "couch", "bed", "bench", "table",
    "t-shirt", "sweater", "jacket", "pants", "shorts",
    "circle", "octagon", "stop sign"
])
file_urls='https://storage.googleapis.com/quickdraw_dataset/sketchrnn/'+files_to_get+'.npz'
file_urls=file_urls.to_list()

In [4]:
xml_url = "https://storage.googleapis.com/quickdraw_dataset?prefix=sketchrnn/"
download_folder = "strokes_data2"
count=0
for url in file_urls:
    count=count+1
    download_file(url, download_folder)
    print(count)
  # downloading the data for which all ablations were done

 Downloading: https://storage.googleapis.com/quickdraw_dataset/sketchrnn/baseball.npz
1
 Downloading: https://storage.googleapis.com/quickdraw_dataset/sketchrnn/basketball.npz
2
 Downloading: https://storage.googleapis.com/quickdraw_dataset/sketchrnn/soccer ball.npz
3
 Downloading: https://storage.googleapis.com/quickdraw_dataset/sketchrnn/wheel.npz
4
 Downloading: https://storage.googleapis.com/quickdraw_dataset/sketchrnn/cookie.npz
5
 Downloading: https://storage.googleapis.com/quickdraw_dataset/sketchrnn/donut.npz
6
 Downloading: https://storage.googleapis.com/quickdraw_dataset/sketchrnn/moon.npz
7
 Downloading: https://storage.googleapis.com/quickdraw_dataset/sketchrnn/clock.npz
8
 Downloading: https://storage.googleapis.com/quickdraw_dataset/sketchrnn/pizza.npz
9
 Downloading: https://storage.googleapis.com/quickdraw_dataset/sketchrnn/cup.npz
10
 Downloading: https://storage.googleapis.com/quickdraw_dataset/sketchrnn/mug.npz
11
 Downloading: https://storage.googleapis.com/quickdra

In [5]:
pwd

'/content'

In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score


gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Using GPU: {gpus}")
    except RuntimeError as e:
        print(e)
else:
    print(" No GPU found. Running on CPU.")


DATA_DIR = "/content/strokes_data2"

files = [f for f in os.listdir(DATA_DIR) if f.endswith(".npz")]
files.sort()

x_train_all, y_train_all, x_vali_all, y_vali_all = [], [], [], []
label_count = 0

for file in files:
    data = np.load(os.path.join(DATA_DIR, file), allow_pickle=True, encoding="latin1")

    x_train = data["train"][:20000]
    x_vali = data["valid"][:20000]

    y_train = [label_count] * len(x_train)
    y_vali = [label_count] * len(x_vali)

    for seq in x_train:
        seq[:, 0] = seq[:, 0].cumsum()
        seq[:, 1] = seq[:, 1].cumsum()
    for seq in x_vali:
        seq[:, 0] = seq[:, 0].cumsum()
        seq[:, 1] = seq[:, 1].cumsum()

    x_train_all.extend(x_train)
    y_train_all.extend(y_train)
    x_vali_all.extend(x_vali)
    y_vali_all.extend(y_vali)

    label_count += 1

print(f"Loaded {label_count} classes.")


MAX_SEQ_LEN = max(len(seq) for seq in x_train_all + x_vali_all)
STROKE_FEATURES = x_train_all[0].shape[1]

X_train = pad_sequences(x_train_all, maxlen=MAX_SEQ_LEN, dtype="float32", padding="post", truncating="post")
X_vali = pad_sequences(x_vali_all, maxlen=MAX_SEQ_LEN, dtype="float32", padding="post", truncating="post")

le = LabelEncoder()
y_train = le.fit_transform(y_train_all)
y_vali = le.transform(y_vali_all)

NUM_CLASSES = len(le.classes_)
y_train = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_vali = tf.keras.utils.to_categorical(y_vali, NUM_CLASSES)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_vali:", X_vali.shape, "y_vali:", y_vali.shape)

#bilstm ablation
def build_bilstm_model(MAX_SEQ_LEN, STROKE_FEATURES, NUM_CLASSES):
    inp = Input(shape=(MAX_SEQ_LEN, STROKE_FEATURES))
    x = Bidirectional(LSTM(128, return_sequences=True))(inp)
    x = Bidirectional(LSTM(64))(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.5)(x)
    out = Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs=inp, outputs=out)

model = build_bilstm_model(MAX_SEQ_LEN, STROKE_FEATURES, NUM_CLASSES)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_vali, y_vali),
    epochs=10,
    batch_size=64
)


val_preds = model.predict(X_vali)
val_preds_labels = np.argmax(val_preds, axis=1)
true_labels = np.argmax(y_vali, axis=1)
acc = accuracy_score(true_labels, val_preds_labels)
print("Final Validation Accuracy:", acc)

train_preds = model.predict(X_train)
train_preds_labels = np.argmax(train_preds, axis=1)
true_labels = np.argmax(y_train, axis=1)
acc = accuracy_score(true_labels, train_preds_labels)
print("Final Training Accuracy:", acc)


✅ Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Loaded 50 classes.
X_train: (1000000, 180, 3) y_train: (1000000, 50)
X_vali: (125000, 180, 3) y_vali: (125000, 50)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 180, 3)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 180, 256)       │       135,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │         3,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 311,026 (1.19 MB)

 Trainable params: 311,026 (1.19 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 534s 34ms/step - accuracy: 0.3508 - loss: 2.1790 - val_accuracy: 0.6060 - val_loss: 1.2061
Epoch 2/10
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 555s 33ms/step - accuracy: 0.5701 - loss: 1.3564 - val_accuracy: 0.6442 - val_loss: 1.0855
Epoch 3/10
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 550s 33ms/step - accuracy: 0.6173 - loss: 1.2089 - val_accuracy: 0.6711 - val_loss: 1.0007
Epoch 4/10
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 508s 32ms/step - accuracy: 0.6417 - loss: 1.1306 - val_accuracy: 0.6872 - val_loss: 0.9470
Epoch 5/10
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 566s 33ms/step - accuracy: 0.6567 - loss: 1.0841 - val_accuracy: 0.6938 - val_loss: 0.9309
Epoch 6/10
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 576s 34ms/step - accuracy: 0.6676 - loss: 1.0467 - val_accuracy: 0.7088 - val_loss: 0.8830
Epoch 7/10
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 526s 34ms/step - accuracy: 0.6773 - loss: 1.0167 - val_accuracy: 0.7084 - val_loss: 0.8812
Epoch 8/10
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 562s 34ms/s